# 04 · Backtesting de Mejoría α

¿La Mejoría α anticipa el bloque siguiente o es regresión a la media? Para cada jugador y partido con Mejoría α calculada a esa fecha, se compara con el cambio de su calificación entre los 5 partidos siguientes y los 5 anteriores. Las conclusiones se informan tal cual salgan, aunque no favorezcan a la métrica.

In [1]:
import sys, statistics, math
sys.path.insert(0, '.')
from datos import api, league_games, result, log_loss, elo_probabilities, m, RESULTS
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [2]:
fechas = {e['match_id']: e['match_date'] for e in api('team_match_master', 'match_id,match_date')}
calif = {}
for r in api('player_match_value', 'match_id,player_id,value', key='eq.rating'):
    if r['value'] is not None and r['match_id'] in fechas:
        calif.setdefault(r['player_id'], []).append((fechas[r['match_id']], r['value']))
for v in calif.values():
    v.sort()
mejoria = api('stat_match_value', 'entity_id,match_id,value', entity='eq.player', stat_id='eq.mejoria_alpha')
pares = []
for r in mejoria:
    serie = calif.get(r['entity_id'], [])
    f = fechas.get(r['match_id'])
    if f is None or r['value'] is None:
        continue
    antes = [c for d, c in serie if d <= f][-5:]
    despues = [c for d, c in serie if d > f][:5]
    if len(antes) >= 3 and len(despues) >= 3:
        pares.append((r['value'], statistics.mean(despues) - statistics.mean(antes), statistics.mean(antes)))
print(f"{len(mejoria)} valores de Mejoría α; {len(pares)} con al menos 3 partidos antes y 3 después")

252 valores de Mejoría α; 202 con al menos 3 partidos antes y 3 después


In [3]:
from scipy.stats import spearmanr
if len(pares) >= 5:
    alfa, cambio, nivel = zip(*pares)
    rho, p = spearmanr(alfa, cambio)
    rho_rm, p_rm = spearmanr(nivel, cambio)
    print(f"Mejoría α contra el cambio siguiente: Spearman {rho:.2f} (p = {p:.2f}, n = {len(pares)})")
    print(f"Nivel previo contra el cambio (regresión a la media): Spearman {rho_rm:.2f} (p = {p_rm:.2f})")
else:
    print('Muestra insuficiente para concluir: hacen falta más partidos con la métrica calculada a la fecha.')

Mejoría α contra el cambio siguiente: Spearman -0.46 (p = 0.00, n = 202)
Nivel previo contra el cambio (regresión a la media): Spearman -0.62 (p = 0.00)


**Lectura.** Una correlación positiva y significativa diría que la Mejoría α anticipa el bloque siguiente; una correlación negativa entre el nivel previo y el cambio es la firma de la regresión a la media. El Potencial α necesita varias temporadas por jugador para validarse: con una sola, queda pendiente.